<a href="https://colab.research.google.com/github/theshambhavipatil/ml4sci_eval/blob/main/task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Loading & Preprocessing

In [ ]:

import h5py
path = r'/kaggle/input/datasets/silentgrad/jetdata/Dataset_Specific_Unlabelled.h5'
with h5py.File(path, 'r') as f:
    print('keys:', list(f.keys()))
    d = f['jet']
    print('shape:', d.shape)
    print('dtype:', d.dtype)
    print('n_samples:', d.shape[0])

    #single sample shape : (125,125,8)

keys: ['jet']
shape: (60000, 125, 125, 8)
dtype: float32
n_samples: 60000


In [ ]:
import h5py
import numpy as np

n_samples=1000 # because of memory constraints i tried to keep this as small as possible
with h5py.File(path, 'r') as f:
    data = f['jet'][:n_samples]

epsilon = 1e-5

#Take the log to compress the massive energy variation
data_log = np.log(data + epsilon)
d_mean = data_log.mean()
d_std = data_log.std()
data_scaled = (data_log - d_mean) / (d_std + 1e-8) # (x - mean) / (std + small val)

Dataset & DataLoader Setup

In [ ]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

class JetDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        x = x.permute(2, 0, 1) # New shape: (8, 125, 125)
        #Pad from 125x125 to 128x128
        x_padded = F.pad(x, (0, 3, 0, 3), mode='constant', value=0)

        return x_padded

# Data Preparation
x_train, x_test = train_test_split(data_scaled, test_size=0.2, random_state=42)

# Create Dataset instances
train_dataset = JetDataset(x_train)
test_dataset = JetDataset(x_test)

# Initialize DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import random

#For reproducability
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#necessary imports

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

 Self-Attention Module

In [ ]:
class selfAttention(nn.Module):

    def __init__(self,n_heads:int,d_embed:int,in_proj: bool = True, out_proj: bool = True):
        super().__init__()

        self.n_heads = n_heads
        self.in_proj=nn.Linear(d_embed,3*d_embed,bias=in_proj)
        self.out_proj=nn.Linear(d_embed,d_embed,bias=out_proj)
        self.d_heads=d_embed//n_heads

    def forward(self,x:torch.Tensor)->torch.Tensor:

        input_shape=x.shape
        batch_size,sequence_length,d_embed=input_shape


        intermim_shape=(batch_size,sequence_length,self.n_heads,self.d_heads)

        q,k,v=self.in_proj(x).chunk(3,dim=-1) #dividing into 3 parts along last dim

        #(batch_size,n_heads,sequence_length,d_heads)
        q=q.view(intermim_shape).transpose(1,2)
        k=k.view(intermim_shape).transpose(1,2)
        v=v.view(intermim_shape).transpose(1,2)

        #(batch_size,n_heads,sequence_length,sequence_length)
        weight=q@k.transpose(-1,-2)

        weight/=math.sqrt(self.d_heads)

        weight=F.softmax(weight,dim=-1)

        #(batch_size,n_heads,sequence_length,sequence_length) @ (batch_size,n_heads,sequence_length,d_heads/n_heads) -> (batch_size,n_heads,sequence_length,d_heads/n_heads)
        output=weight@v

        output = output.transpose(1, 2).contiguous()
        output = output.view(input_shape)
        output = self.out_proj(output)

        return output

UNET

In [ ]:
class UNET_AttentionBlock(nn.Module):
    '''
    geglu activation function i have chosen to use from the paper,
    i have used it to stay close to the original architecture of the
    paper
     '''

    def __init__(self,n_heads:int,n_embed:int):

        super().__init__()
        channels=n_heads*n_embed # all heads have context of all embeddings

        self.groupnorm = nn.GroupNorm(min(32, channels), channels)
        self.conv_inp=nn.Conv2d(channels,channels,kernel_size=1,padding=0)

        self.layernorm1=nn.LayerNorm(channels)
        self.attention1=selfAttention(n_heads,channels,in_proj=False, out_proj=True)
        self.layernorm2=nn.LayerNorm(channels)
        self.linear_geglu1=nn.Linear(channels,4*channels*2)

        self.linear_geglu2=nn.Linear(4*channels,channels)

        self.conv_out=nn.Conv2d(channels,channels,kernel_size=1,padding=0)

    def forward(self,x):
        #x : (batch_size,channels,height,width)

        residue_long=x
        x=self.groupnorm(x)
        x=self.conv_inp(x)
        n,c,h,w=x.shape

        # Flatten spatial dimensions for attention
        x=x.view(n,c,h*w).transpose(-1,-2)

        #self attention
        residue_short=x
        x=self.layernorm1(x)
        x=self.attention1(x)
        x=x+residue_short
        residue_short=x

        #feed forward
        x=self.layernorm2(x)
        x,gate=self.linear_geglu1(x).chunk(2,dim=-1)
        x=x*F.gelu(gate)

        x=self.linear_geglu2(x)
        x=x+residue_short

        x=x.transpose(-1,-2).view(n,c,h,w)
        return self.conv_out(x) +residue_long

class UNET_ResidualBlock(nn.Module):

    def __init__(self,in_channels:int,out_channels:int,time:int=256):
        super().__init__()

        self.groupnorm_feature = nn.GroupNorm(min(32, in_channels), in_channels)
        self.conv_feature=nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1)
        self.linear_time=nn.Linear(time,out_channels)

        self.groupnorm_2 = nn.GroupNorm(min(32, out_channels), out_channels)
        self.conv_2=nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1)

        if in_channels == out_channels:
            self.residual_connection=nn.Identity()
        else:
            self.residual_connection=nn.Conv2d(in_channels,out_channels,kernel_size=1)

    def forward(self,feature,time)->torch.Tensor:
        #feature :(batch_size,channels,height,width)
        #time: (1,time_emb)

        residue=feature

        feature=self.groupnorm_feature(feature)
        feature=F.silu(feature)
        feature=self.conv_feature(feature)

        time=F.silu(time)
        time=self.linear_time(time)

        merged=feature+time.unsqueeze(-1).unsqueeze(-1)
        merged=self.groupnorm_2(merged)
        merged=F.silu(merged)
        merged=self.conv_2(merged)

        return merged+self.residual_connection(residue)

class SwitchSequential(nn.Sequential):
    '''
    used to decide what input parameters each block receives
    '''

    def forward(self,x:torch.tensor,time:torch.tensor)->torch.tensor:
        for layer in self:
            if isinstance(layer,UNET_AttentionBlock):
                x=layer(x)
            elif isinstance(layer,UNET_ResidualBlock):
                x=layer(x,time)
            else:
                x=layer(x)
        return x

class TimeEmbedding(nn.Module):
    def __init__(self, dim=320):
        super().__init__()
        self.half_dim = dim // 2

    def forward(self, timestep: torch.Tensor):
        device = timestep.device
        freqs = torch.pow(10000, -torch.arange(start=0, end=self.half_dim, dtype=torch.float32, device=device) / self.half_dim)
        x = timestep[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(x), torch.cos(x)], dim=-1)

In [ ]:

class UNET(nn.Module):
    def __init__(self, in_channels=8, out_channels=8):
        super().__init__()
        self.time_dim = 320

        self.time_mlp = nn.Sequential(
            TimeEmbedding(dim=self.time_dim),
            nn.Linear(self.time_dim, self.time_dim),
            nn.GELU(),
            nn.Linear(self.time_dim, self.time_dim)
        )

        # encoder

        # l1_enc  8->32 (128*128)
        self.enc1 = SwitchSequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            UNET_ResidualBlock(32, 32, time=self.time_dim)
        )

        # l2_enc: 32->64 (64*64)
        self.enc2 = SwitchSequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            UNET_ResidualBlock(64, 64, time=self.time_dim)
        )

        # l3_enc: 64->128 (32*32)
        self.enc3 = SwitchSequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            UNET_ResidualBlock(128, 128, time=self.time_dim),
            UNET_AttentionBlock(n_heads=4, n_embed=32)
        )

        #bottlenceck layer

        # no change in channels or size ( as you can observe we are bringing the attention block in the bottom layer can see every "pixel" with less memory requires)
        self.bottleneck = SwitchSequential(
            UNET_ResidualBlock(128, 128, time=self.time_dim),
            UNET_AttentionBlock(n_heads=4, n_embed=32),
            UNET_ResidualBlock(128, 128, time=self.time_dim)
        )

        # decoder
        # 128 (bottleneck)+128 (l3_enc)
        self.dec3 = SwitchSequential(
            UNET_ResidualBlock(256, 64, time=self.time_dim),
            UNET_AttentionBlock(n_heads=4, n_embed=16),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        )

        # 64 (l3_dec)+64 (l2_enc)
        self.dec2 = SwitchSequential(
            UNET_ResidualBlock(128, 32, time=self.time_dim),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        )

        #  32 (l2_dec)+32 (l1_enc)
        self.dec1 = SwitchSequential(
            UNET_ResidualBlock(64, 32, time=self.time_dim)
        )

        # Output projection
        self.out = nn.Sequential(
            nn.GroupNorm(min(16, 32), 32),
            nn.SiLU(),
            nn.Conv2d(32, out_channels, kernel_size=3, padding=1)
        )

    def forward(self, x: torch.Tensor, time: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_mlp(time)

        skip1 = self.enc1(x, t_emb)
        skip2 = self.enc2(skip1, t_emb)
        skip3 = self.enc3(skip2, t_emb)

        x = self.bottleneck(skip3, t_emb)

        x = torch.cat([x, skip3], dim=1)
        x = self.dec3(x, t_emb)

        x = torch.cat([x, skip2], dim=1)
        x = self.dec2(x, t_emb)

        x = torch.cat([x, skip1], dim=1)
        x = self.dec1(x, t_emb)

        return self.out(x)


DDPM module(sampler and scheduler)

In [ ]:
import torch
import numpy as np

# i have heavily reffered to this repository https://github.com/lucidrains/denoising-diffusion-pytorch
#and also referred the orig research paper for the math(especially for this part forward process P(X_t|X_t-1) and backward process P(X_t-1|X_t)

class DDPMSampler:
    def __init__(self, generator: torch.Generator = None, num_training_steps=1000, beta_start=0.0001, beta_end=0.02):
        # A standard linear schedule is usually highly effective for non-image physics data
        self.betas = torch.linspace(beta_start, beta_end, num_training_steps, dtype=torch.float32)

        # Forward process variables
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.one = torch.tensor(1.0)

        self.generator = generator
        self.num_training_steps = num_training_steps

        # Timesteps in reverse order (T to 0)
        self.timesteps = torch.from_numpy(np.arange(0, num_training_steps)[::-1].copy())

    def set_inference_timesteps(self, num_inference_steps: int = 50):
        self.num_inference_steps = num_inference_steps
        step_ratio = self.num_training_steps // self.num_inference_steps

        timesteps = (np.arange(0, num_inference_steps) * step_ratio).round()[::-1].copy().astype(np.int64)
        self.timesteps = torch.tensor(timesteps)

    def _get_previous_timestep(self, timestep: int) -> int:
        return timestep - (self.num_training_steps // self.num_inference_steps)

    def add_noise(self, original_samples: torch.Tensor, timesteps: torch.Tensor):
        """
        forward process: add noise to the data
        """
        alphas_cumprod = self.alphas_cumprod.to(device=original_samples.device, dtype=original_samples.dtype)
        timesteps = timesteps.to(original_samples.device)

        sqrt_alpha_prod = alphas_cumprod[timesteps]**0.5
        sqrt_one_minus_alpha_prod = (1 - alphas_cumprod[timesteps])**0.5

        #(b,c,h,w)
        sqrt_alpha_prod = sqrt_alpha_prod.view(-1,1,1,1)
        sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.view(-1,1,1,1)

        # Generate the random noise N(0, I)
        noise = torch.randn(
            original_samples.shape,
            generator=self.generator,
            device=original_samples.device,
            dtype=original_samples.dtype
        )

        noisy_samples=(sqrt_alpha_prod*original_samples)+(sqrt_one_minus_alpha_prod* noise)

        return noisy_samples,noise

    @torch.no_grad()
    def step(self,timestep:int,latents:torch.Tensor,model_output:torch.Tensor)->torch.Tensor:
        """
        reverse process: denoises the data by one timestep
        """
        t = timestep
        prev_t = self._get_previous_timestep(t)

        # Move schedule constants to the correct device
        device=latents.device
        alpha_prod_t=self.alphas_cumprod[t].to(device)
        alpha_prod_t_prev=self.alphas_cumprod[prev_t].to(device) if prev_t>= 0 else self.one.to(device)

        beta_prod_t=1-alpha_prod_t
        beta_prod_t_prev=1-alpha_prod_t_prev
        current_alpha_t=alpha_prod_t/alpha_prod_t_prev
        current_beta_t=1-current_alpha_t

        # Predict original sample x_0 from the UNet's predicted noise
        pred_original_sample=(latents-beta_prod_t**(0.5)*model_output)/alpha_prod_t**(0.5)

        # Compute coefficients for the posterior mean
        pred_original_sample_coeff=(alpha_prod_t_prev**(0.5)*current_beta_t)/beta_prod_t
        current_sample_coeff=current_alpha_t**(0.5)*beta_prod_t_prev/beta_prod_t

        # Compute predicted previous sample mean (mu_t)
        pred_prev_sample=pred_original_sample_coeff *pred_original_sample+current_sample_coeff*latents

        # Add variance if not at t=0 as (orig image doesnt have noise at timestep=0)
        variance = 0
        if t > 0:
            noise = torch.randn(
                model_output.shape,
                generator=self.generator,
                device=device,
                dtype=model_output.dtype
            )
            # Compute variance (Formula 7 from DDPM paper)
            variance_val = (1 - alpha_prod_t_prev)/(1 - alpha_prod_t)*current_beta_t
            variance_val = torch.clamp(variance_val,min=1e-20)
            variance = (variance_val** 0.5)* noise

        return pred_prev_sample + variance

training loop

In [ ]:
from tqdm import tqdm

model = UNET(in_channels=8, out_channels=8).to(device)
noise_scheduler = DDPMSampler(num_training_steps=1000)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()
epochs = 50

best_loss = float('inf')
save_path = "/kaggle/working/unet_model.pth"

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        batch = batch.to(device)

        # Sample random timesteps for the batch
        t = torch.randint(0,noise_scheduler.num_training_steps,(batch.shape[0],),device=device).long()
        # Forward Process (add noise)
        noisy_batch, noise = noise_scheduler.add_noise(batch,t)
        # Predict the noise using the U-Net (params: noisy iamge ,time_step)
        pred_noise = model(noisy_batch,t)
        # Calculate loss and update weights
        loss = criterion(pred_noise,noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss +=loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), save_path)
print(f"Training complete! Final model saved to {save_path}")

Epoch 1/50: 100%|██████████| 50/50 [00:11<00:00,  4.28it/s]


Epoch 1 Average Loss: 0.9193


Epoch 2/50: 100%|██████████| 50/50 [00:11<00:00,  4.18it/s]


Epoch 2 Average Loss: 0.6790


Epoch 3/50: 100%|██████████| 50/50 [00:12<00:00,  4.08it/s]


...

Epoch 48 Average Loss: 0.0692


Epoch 49/50: 100%|██████████| 50/50 [00:11<00:00,  4.22it/s]


Epoch 49 Average Loss: 0.0641


Epoch 50/50: 100%|██████████| 50/50 [00:11<00:00,  4.22it/s]

Epoch 50 Average Loss: 0.0623
Training complete! Final model saved to /kaggle/working/unet_model.pth


**Conclusion**

wrapping up task 1.
Initially i applied a log transforamtion(high skew ) and then z transformation(normal distribution as can be seen normally in CMS collider data or any physical event to talk about).Ensure this could run smoothly on kaggle free tier i created a custom U-NET.Padded the images (125x125->128x128) for clean convolutions and trained the model using a batch size of 16 and a learning rate of 1e-4.For training number of epochs were 50
across 1000 diffusion timesteps and 50 step reverse proces.